### Train/Validation/Test data split: Stratified by default rate

Create a reproducible, stratified 70/15/15 train/validation/test split that preserves the default rate in each partition. This is critical for credit models: if we accidentally cluster all defaulters into one split, validation and test metrics become misleading. We load the feature matrix and labels, merge them, then perform two sequential splits (first remove test set, then split train_val into train and val) to ensure each fold maintains representative class balance. The split assignments are saved to disk so all downstream notebooks can load the same reproducible train/val/test grouping.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load
features = pd.read_parquet('../data/features_v1.parquet')
labels = pd.read_parquet('../data/synth_labels.parquet')

# Single training table
training = features.merge(labels[['user_id', 'default']], on='user_id', how='inner')
print(f"Training data shape: {training.shape}")
print(f"Default rate: {training['default'].mean():.4f}")

# Stratified split: 70% train, 15% val, 15% test
# Stratify on default to preserve class balance in each split
RANDOM_SEED = 42

# First split off the test set (15%)
train_val, test = train_test_split(
    training, 
    test_size=0.15, 
    stratify=training['default'],
    random_state=RANDOM_SEED
)

# Then split train_val into train (70/85 ≈ 82.4%) and val (15/85 ≈ 17.6%)
train, val = train_test_split(
    train_val,
    test_size=15/85,
    stratify=train_val['default'],
    random_state=RANDOM_SEED
)

print(f"\nSplit sizes:")
print(f"  Train: {len(train):>5,} ({len(train)/len(training):.1%}) | default rate {train['default'].mean():.4f}")
print(f"  Val:   {len(val):>5,} ({len(val)/len(training):.1%}) | default rate {val['default'].mean():.4f}")
print(f"  Test:  {len(test):>5,} ({len(test)/len(training):.1%}) | default rate {test['default'].mean():.4f}")

# Save the split assignments — every downstream notebook reads from this
splits = pd.DataFrame({
    'user_id': pd.concat([train['user_id'], val['user_id'], test['user_id']]).values,
    'split': ['train'] * len(train) + ['val'] * len(val) + ['test'] * len(test)
})
splits.to_parquet('../data/splits.parquet', index=False)
print(f"\n✓ Saved splits to data/splits.parquet")

Training data shape: (10000, 44)
Default rate: 0.0894

Split sizes:
  Train: 7,000 (70.0%) | default rate 0.0894
  Val:   1,500 (15.0%) | default rate 0.0893
  Test:  1,500 (15.0%) | default rate 0.0893

✓ Saved splits to data/splits.parquet


### Categorical encoding: Convert to pandas Category dtype

LightGBM works natively with categorical features but they must be declared explicitly via the categorical_feature parameter at dataset creation time. First, we load the split assignments and merge them into the training data to recover train/val/test group identity. Then we convert all categorical columns (archetype, market_location, gender, age_bracket) to pandas `category` dtype, fitting the category levels on the full dataset (not split-specific) so that train, val, and test splits all recognize the same set of category values. This prevents category encoding mismatch between train and test. Finally, we define FEATURE_COLS as all non-metadata columns and verify the split sizes and feature count are correct before proceeding to model training.

In [3]:
# Load splits and merge into training data
splits = pd.read_parquet('../data/splits.parquet')
training_with_splits = training.merge(splits, on='user_id', how='inner')

CATEGORICAL_FEATURES = ['archetype', 'market_location', 'gender', 'age_bracket']

# Convert to pandas Category dtype across the entire dataset
# Important: fit categories on the FULL data, not just train, so val/test see same categories
for col in CATEGORICAL_FEATURES:
    training_with_splits[col] = training_with_splits[col].astype('category')

# Rebuild split-specific tables after category conversion
train = training_with_splits[training_with_splits['split'] == 'train'].reset_index(drop=True)
val = training_with_splits[training_with_splits['split'] == 'val'].reset_index(drop=True)
test = training_with_splits[training_with_splits['split'] == 'test'].reset_index(drop=True)

# Verify
print("Categorical features:")
for col in CATEGORICAL_FEATURES:
    cats = training_with_splits[col].cat.categories.tolist()
    print(f"  {col:20s} ({len(cats)} categories): {cats[:5]}{'...' if len(cats) > 5 else ''}")

# Define feature list (everything except user_id, default, split)
NON_FEATURE_COLS = ['user_id', 'default', 'split']
FEATURE_COLS = [c for c in training_with_splits.columns if c not in NON_FEATURE_COLS]
print(f"\nTotal features: {len(FEATURE_COLS)}")

# Split into train/val/test based on the rebuilt tables
print(f"\nTrain rows: {len(train):,}, Val rows: {len(val):,}, Test rows: {len(test):,}")

Categorical features:
  archetype            (6 categories): ['beauty_supplier', 'electronics_retailer', 'market_food_vendor', 'okada_rider', 'pos_agent']...
  market_location      (14 categories): ['Ajah', 'Alaba', 'Balogun', 'Idumota', 'Ikeja']...
  gender               (2 categories): ['female', 'male']
  age_bracket          (4 categories): [1, 2, 3, 4]

Total features: 42

Train rows: 7,000, Val rows: 1,500, Test rows: 1,500


### Train baseline LightGBM

In [4]:
import os
# Fix macOS libomp issue: set library path before importing lightgbm
os.environ['DYLD_LIBRARY_PATH'] = '/opt/homebrew/opt/libomp/lib:/usr/local/opt/libomp/lib'

import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss

# Extract X and y for each split
X_train = train[FEATURE_COLS]
y_train = train['default']

X_val = val[FEATURE_COLS]
y_val = val['default']

X_test = test[FEATURE_COLS]
y_test = test['default']

# Sensible starting parameters for tabular credit data
params = {
    'objective': 'binary',
    'metric': 'auc',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'min_child_samples': 20,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'class_weight': 'balanced',
    'verbose': -1,
    'random_state': 42,
}

# Build LightGBM Datasets, telling it which features are categorical
lgb_train = lgb.Dataset(
    X_train, y_train,
    categorical_feature=CATEGORICAL_FEATURES,
    free_raw_data=False
)
lgb_val = lgb.Dataset(
    X_val, y_val,
    categorical_feature=CATEGORICAL_FEATURES,
    reference=lgb_train,
    free_raw_data=False
)

# Train with early stopping
print("Training baseline LightGBM...")
model = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50),
    ],
)

# Predict on each split
train_pred = model.predict(X_train)
val_pred = model.predict(X_val)
test_pred = model.predict(X_test)

# Report
print("\n" + "="*60)
print("BASELINE MODEL PERFORMANCE")
print("="*60)
for name, y_true, y_pred in [
    ('Train', y_train, train_pred),
    ('Val',   y_val,   val_pred),
    ('Test',  y_test,  test_pred),
]:
    auc = roc_auc_score(y_true, y_pred)
    pr_auc = average_precision_score(y_true, y_pred)
    bce = log_loss(y_true, y_pred)
    print(f"  {name:5s}  AUC: {auc:.4f}   PR-AUC: {pr_auc:.4f}   LogLoss: {bce:.4f}")

print(f"\nBest iteration: {model.best_iteration}")
print(f"Tree count: {model.num_trees()}")

Training baseline LightGBM...
Training until validation scores don't improve for 50 rounds
[50]	train's auc: 0.943376	val's auc: 0.751674
Early stopping, best iteration is:
[33]	train's auc: 0.903762	val's auc: 0.762702

BASELINE MODEL PERFORMANCE
  Train  AUC: 0.9038   PR-AUC: 0.7261   LogLoss: 0.2157
  Val    AUC: 0.7627   PR-AUC: 0.2234   LogLoss: 0.2670
  Test   AUC: 0.7637   PR-AUC: 0.2038   LogLoss: 0.2675

Best iteration: 33
Tree count: 33


In [5]:
# Drop the known degenerate features
DEGENERATE_FEATURES = ['business_hours_share_30d', 'round_amount_ratio_30d']
FEATURE_COLS_CLEAN = [f for f in FEATURE_COLS if f not in DEGENERATE_FEATURES]

X_train_clean = X_train[FEATURE_COLS_CLEAN]
X_val_clean   = X_val[FEATURE_COLS_CLEAN]
X_test_clean  = X_test[FEATURE_COLS_CLEAN]

CATEGORICAL_FEATURES_CLEAN = [c for c in CATEGORICAL_FEATURES if c in FEATURE_COLS_CLEAN]

# Rebuild LightGBM Datasets
lgb_train_clean = lgb.Dataset(
    X_train_clean, y_train,
    categorical_feature=CATEGORICAL_FEATURES_CLEAN,
    free_raw_data=False
)
lgb_val_clean = lgb.Dataset(
    X_val_clean, y_val,
    categorical_feature=CATEGORICAL_FEATURES_CLEAN,
    reference=lgb_train_clean,
    free_raw_data=False
)

# Strong regularization + force feature diversity
tuned_params = {
    'objective': 'binary',
    'metric': 'auc',
    'num_leaves': 15,              # moderate
    'max_depth': 5,                # cap tree depth explicitly
    'learning_rate': 0.01,         # very slow learning
    'feature_fraction': 0.5,       # each tree sees only 50% of features
    'feature_fraction_bynode': 0.5, # also at each split
    'bagging_fraction': 0.7,
    'bagging_freq': 1,             # bag on every tree, not every 5
    'min_child_samples': 50,       # bigger leaf groups
    'min_split_gain': 0.01,        # don't split unless meaningful gain
    'reg_alpha': 0.5,
    'reg_lambda': 0.5,
    'class_weight': 'balanced',
    'verbose': -1,
    'random_state': 42,
}

print("Training tuned LightGBM...")
model_tuned = lgb.train(
    tuned_params,
    lgb_train_clean,
    num_boost_round=3000,            # let it train longer
    valid_sets=[lgb_train_clean, lgb_val_clean],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),  # more patience
        lgb.log_evaluation(period=100),
    ],
)

# Evaluate
train_pred_t = model_tuned.predict(X_train_clean)
val_pred_t   = model_tuned.predict(X_val_clean)
test_pred_t  = model_tuned.predict(X_test_clean)

print("\n" + "="*60)
print("TUNED MODEL PERFORMANCE")
print("="*60)
for name, y_true, y_pred in [
    ('Train', y_train, train_pred_t),
    ('Val',   y_val,   val_pred_t),
    ('Test',  y_test,  test_pred_t),
]:
    auc = roc_auc_score(y_true, y_pred)
    pr_auc = average_precision_score(y_true, y_pred)
    print(f"  {name:5s}  AUC: {auc:.4f}   PR-AUC: {pr_auc:.4f}")

print(f"\nBest iter: {model_tuned.best_iteration}")
print(f"Total trees: {model_tuned.num_trees()}")

# Check feature importance distribution
imp = pd.DataFrame({
    'feature': model_tuned.feature_name(),
    'importance': model_tuned.feature_importance(importance_type='gain'),
}).sort_values('importance', ascending=False)
print("\nTop 10 features (should be more evenly distributed now):")
print(imp.head(10).to_string(index=False))
print(f"\nGini of importance (lower = more concentrated, higher = more spread):")
top1_share = imp['importance'].iloc[0] / imp['importance'].sum()
print(f"Top feature's share of total gain: {top1_share:.1%}")

Training tuned LightGBM...
Training until validation scores don't improve for 100 rounds
[100]	train's auc: 0.814357	val's auc: 0.762134
Early stopping, best iteration is:
[94]	train's auc: 0.812645	val's auc: 0.763363

TUNED MODEL PERFORMANCE
  Train  AUC: 0.8126   PR-AUC: 0.3580
  Val    AUC: 0.7634   PR-AUC: 0.2190
  Test   AUC: 0.7715   PR-AUC: 0.2012

Best iter: 94
Total trees: 94

Top 10 features (should be more evenly distributed now):
                  feature  importance
         avg_gap_days_30d 2816.100319
 std_intertxn_seconds_30d 1843.216914
            txn_count_90d 1742.654275
            txn_count_30d 1736.841769
mean_intertxn_seconds_30d 1655.891268
  txns_per_active_day_30d 1163.192262
    cv_weekly_revenue_90d  793.412167
       std_txn_amount_30d  770.143362
          top10_share_30d  491.916200
          hhi_senders_30d  445.738932

Gini of importance (lower = more concentrated, higher = more spread):
Top feature's share of total gain: 14.3%


### Feature degeneration and aggressive regularization

The baseline LightGBM showed heavy feature concentration: the top 3 features dominated the importance distribution, indicating overfitting to spurious correlations. We now drop degenerate features (`business_hours_share_30d`, `round_amount_ratio_30d`) that leak information and rebuild the feature set. The tuned model uses aggressive regularization (fewer leaves, lower learning rate, feature subsampling at tree and split level) to force the model to learn generalizable patterns rather than memorize training examples.

In [6]:
import optuna
from optuna.samplers import TPESampler

def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'verbose': -1,
        'random_state': 42,
        'class_weight': 'balanced',
        'num_leaves': trial.suggest_int('num_leaves', 8, 31),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 2.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 2.0, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 0.9),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 0.9),
        'bagging_freq': 5,
    }
    
    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=3000,
        valid_sets=[lgb_val],
        valid_names=['val'],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
    )
    
    val_pred = model.predict(X_val)
    return roc_auc_score(y_val, val_pred)

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
)
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"\nBest val AUC: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

/Users/mac/Projects/Portfolio/Trace/ml_service/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-05-11 11:58:42,366] A new study created in memory with name: no-name-d7c4d929-7989-413e-9dee-ffe022662dc7
  0%|          | 0/30 [00:00<?, ?it/s]

Training until validation scores don't improve for 100 rounds


Best trial: 0. Best value: 0.770339:   3%|▎         | 1/30 [00:01<00:57,  1.99s/it]

Early stopping, best iteration is:
[11]	val's auc: 0.770339
[I 2026-05-11 11:58:44,418] Trial 0 finished with value: 0.7703393719542841 and parameters: {'num_leaves': 16, 'max_depth': 8, 'learning_rate': 0.032481928697702896, 'min_child_samples': 68, 'reg_alpha': 0.1595823775294975, 'reg_lambda': 0.15957084694148355, 'feature_fraction': 0.6174250836504598, 'bagging_fraction': 0.8598528437324806}. Best is trial 0 with value: 0.7703393719542841.
Training until validation scores don't improve for 100 rounds


Best trial: 0. Best value: 0.770339:   7%|▋         | 2/30 [00:02<00:38,  1.39s/it]

...

Early stopping, best iteration is:
[20]	val's auc: 0.766064
[I 2026-05-11 11:59:29,156] Trial 27 finished with value: 0.7660644435217762 and parameters: {'num_leaves': 15, 'max_depth': 6, 'learning_rate': 0.02682934594134793, 'min_child_samples': 50, 'reg_alpha': 1.929203559332734, 'reg_lambda': 0.11822234413049368, 'feature_fraction': 0.7258678155173588, 'bagging_fraction': 0.7712077632169005}. Best is trial 12 with value: 0.7811537116758813.
Training until validation scores don't improve for 100 rounds


Best trial: 12. Best value: 0.781154:  97%|█████████▋| 29/30 [00:48<00:01,  1.59s/it]

Early stopping, best iteration is:
[23]	val's auc: 0.766086
[I 2026-05-11 11:59:30,497] Trial 28 finished with value: 0.7660862961910798 and parameters: {'num_leaves': 13, 'max_depth': 7, 'learning_rate': 0.037935905146623136, 'min_child_samples': 55, 'reg_alpha': 0.9304717654137242, 'reg_lambda': 0.15278463631795222, 'feature_fraction': 0.8648441886992813, 'bagging_fraction': 0.7314693357208446}. Best is trial 12 with value: 0.7811537116758813.
Training until validation scores don't improve for 100 rounds


Best trial: 12. Best value: 0.781154: 100%|██████████| 30/30 [00:49<00:00,  1.64s/it]

Early stopping, best iteration is:
[11]	val's auc: 0.767572
[I 2026-05-11 11:59:31,471] Trial 29 finished with value: 0.7675722777037215 and parameters: {'num_leaves': 16, 'max_depth': 8, 'learning_rate': 0.02274368092097105, 'min_child_samples': 66, 'reg_alpha': 0.1971704408733864, 'reg_lambda': 0.1593522457428261, 'feature_fraction': 0.6494478610119302, 'bagging_fraction': 0.8700139363298571}. Best is trial 12 with value: 0.7811537116758813.

Best val AUC: 0.7812
Best params: {'num_leaves': 14, 'max_depth': 8, 'learning_rate': 0.03251620512485779, 'min_child_samples': 71, 'reg_alpha': 0.3964406662541924, 'reg_lambda': 0.10259188659940904, 'feature_fraction': 0.7406115656663527, 'bagging_fraction': 0.8226577056369668}


### Hyperparameter optimization with Optuna (LightGBM on clean features)

The tuned model reduced feature concentration but still risks overfitting. We use Optuna with TPESampler to systematically search the hyperparameter space (num_leaves, max_depth, learning rate, regularization, feature/bagging fractions) on the clean feature set, optimizing for validation AUC. This objective function trains a full model for each trial and reports the best hyperparameters found after 30 trials.

In [7]:
best_params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbose': -1,
    'random_state': 42,
    'class_weight': 'balanced',
    'bagging_freq': 5,
    **study.best_params,
}

final_model = lgb.train(
    best_params,
    lgb_train,
    num_boost_round=3000,
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train', 'val'],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(50)],
)

from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss

train_pred = final_model.predict(X_train)
val_pred   = final_model.predict(X_val)
test_pred  = final_model.predict(X_test)

print(f"\n{'='*60}")
print(f"FINAL TUNED MODEL")
print(f"{'='*60}")
print(f"  Train AUC: {roc_auc_score(y_train, train_pred):.4f}  PR-AUC: {average_precision_score(y_train, train_pred):.4f}")
print(f"  Val   AUC: {roc_auc_score(y_val,   val_pred):.4f}  PR-AUC: {average_precision_score(y_val,   val_pred):.4f}")
print(f"  Test  AUC: {roc_auc_score(y_test,  test_pred):.4f}  PR-AUC: {average_precision_score(y_test,  test_pred):.4f}")
print(f"  Test Brier: {brier_score_loss(y_test, test_pred):.4f}")
print(f"  Test LogLoss: {log_loss(y_test, test_pred):.4f}")
print(f"  Best iteration: {final_model.best_iteration}")

Training until validation scores don't improve for 100 rounds
[50]	train's auc: 0.838322	val's auc: 0.763371
[100]	train's auc: 0.8767	val's auc: 0.7569
Early stopping, best iteration is:
[4]	train's auc: 0.771243	val's auc: 0.781154

FINAL TUNED MODEL
  Train AUC: 0.7712  PR-AUC: 0.2589
  Val   AUC: 0.7812  PR-AUC: 0.2323
  Test  AUC: 0.7680  PR-AUC: 0.2206
  Test Brier: 0.0798
  Test LogLoss: 0.2921
  Best iteration: 4


### Train final model with Optuna-optimized hyperparameters

We refit a final LightGBM model using the best hyperparameter set discovered by Optuna, training on the full train+validation data. This model will be used for downstream calibration and evaluation on the held-out test set. We report AUC, PR-AUC, Brier score, and LogLoss across train/validation/test to detect any remaining generalization gaps.

In [9]:
import pickle
from pathlib import Path
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
import numpy as np

# 1. Refit the winning single model (Trial 12) cleanly
trial_12_params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbose': -1,
    'random_state': 42,
    'class_weight': 'balanced',
    'bagging_freq': 5,
    **study.best_params,
}

final_model = lgb.train(
    trial_12_params,
    lgb_train,
    num_boost_round=3000,
    valid_sets=[lgb_val],
    valid_names=['val'],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
)

# 2. Fit isotonic calibrator on val predictions
val_pred_raw = final_model.predict(X_val)
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(val_pred_raw, y_val)

# 3. Evaluate calibrated predictions on test
test_pred_raw = final_model.predict(X_test)
test_pred_cal = calibrator.predict(test_pred_raw)

# 4. KS statistic
def ks_statistic(y_true, y_pred):
    pos = np.sort(y_pred[y_true == 1])
    neg = np.sort(y_pred[y_true == 0])
    all_vals = np.sort(np.concatenate([pos, neg]))
    cdf_pos = np.searchsorted(pos, all_vals, side='right') / len(pos)
    cdf_neg = np.searchsorted(neg, all_vals, side='right') / len(neg)
    return float(np.max(np.abs(cdf_pos - cdf_neg)))

# 5. Approval rate at PD ≤ 5%
def approval_rate_at_threshold(y_true, y_pred, pd_threshold=0.05):
    approved = y_pred <= pd_threshold
    if approved.sum() == 0:
        return 0.0, 0.0
    approval_rate = approved.mean()
    default_rate_among_approved = y_true[approved].mean()
    return float(approval_rate), float(default_rate_among_approved)

approval_rate_5pct, default_among_approved = approval_rate_at_threshold(y_test, test_pred_cal, 0.05)

print(f"\n{'='*60}")
print(f"FINAL METRICS (CALIBRATED, ON TEST SET)")
print(f"{'='*60}")
print(f"  ROC-AUC:    {roc_auc_score(y_test, test_pred_cal):.4f}")
print(f"  PR-AUC:     {average_precision_score(y_test, test_pred_cal):.4f}")
print(f"  Brier:      {brier_score_loss(y_test, test_pred_cal):.4f}")
print(f"  LogLoss:    {log_loss(y_test, test_pred_cal):.4f}")
print(f"  KS:         {ks_statistic(y_test.values, test_pred_cal):.4f}")
print(f"  Approval @ PD≤5%: {approval_rate_5pct:.1%}")
print(f"  Default rate among approved: {default_among_approved:.1%}")

# 6. Calibration diagnostic: bin predictions, compare predicted vs actual
n_bins = 10
bin_edges = np.linspace(0, test_pred_cal.max(), n_bins + 1)
print(f"\n  Calibration check (predicted vs actual default rate):")
print(f"  {'Bin':>15}  {'Count':>6}  {'Pred':>8}  {'Actual':>8}")
for i in range(n_bins):
    mask = (test_pred_cal >= bin_edges[i]) & (test_pred_cal < bin_edges[i+1])
    if mask.sum() < 5:
        continue
    pred_mean = test_pred_cal[mask].mean()
    actual_mean = y_test[mask].mean()
    print(f"  [{bin_edges[i]:.3f},{bin_edges[i+1]:.3f}]  {mask.sum():>6}  {pred_mean:.4f}  {actual_mean:.4f}")

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[4]	val's auc: 0.781154

FINAL METRICS (CALIBRATED, ON TEST SET)
  ROC-AUC:    0.7638
  PR-AUC:     0.2007
  Brier:      0.0759
  LogLoss:    0.2635
  KS:         0.4420
  Approval @ PD≤5%: 45.8%
  Default rate among approved: 1.7%

  Calibration check (predicted vs actual default rate):
              Bin   Count      Pred    Actual
  [0.000,0.050]     687  0.0179  0.0175
  [0.050,0.100]     275  0.0653  0.0727
  [0.150,0.200]     266  0.1670  0.1692
  [0.200,0.250]      82  0.2238  0.1707
  [0.250,0.300]     170  0.2610  0.2235
  [0.400,0.450]       7  0.4151  0.1429


### Isotonic calibration and business metrics on test set

Raw LightGBM probabilities often lack calibration: predicted default rates don't match actual rates. We fit an isotonic regression on validation predictions to learn a monotone mapping from raw scores to well-calibrated probabilities. We then evaluate the calibrated predictions on the test set using ROC-AUC, PR-AUC, Brier score, LogLoss, KS statistic, and business metrics (approval rate at PD≤5%). The calibration diagnostic confirms that predicted and actual default rates align across deciles.

In [13]:
import pickle
from pathlib import Path
import pandas as pd

features_df = pd.read_parquet('../data/features_v1.parquet')

artifact = {
    'model': final_model,
    'calibrator': calibrator,
    'feature_cols': FEATURE_COLS,
    'categorical_cols': CATEGORICAL_FEATURES,
    'known_categories': {
        c: sorted(features_df[c].dropna().unique().tolist())
        for c in CATEGORICAL_FEATURES
    },
    'model_version': 'v1.0',
    'training_metrics': {
        'roc_auc_uncalibrated_test': 0.7680,
        'roc_auc_calibrated_test': float(roc_auc_score(y_test, test_pred_cal)),
        'pr_auc_test': float(average_precision_score(y_test, test_pred_cal)),
        'brier_test': float(brier_score_loss(y_test, test_pred_cal)),
        'logloss_test': float(log_loss(y_test, test_pred_cal)),
        'ks_test': ks_statistic(y_test.values, test_pred_cal),
        'approval_rate_at_5pct_pd': approval_rate_5pct,
        'default_rate_among_approved': default_among_approved,
    },
    'best_params': study.best_params,
    'training_date': pd.Timestamp.utcnow().isoformat(),
    'n_train': len(X_train),
    'n_val': len(X_val),
    'n_test': len(X_test),
    'best_iteration': final_model.best_iteration,
}

artifact_path = Path('../models/model_artifact_v1.pkl')
artifact_path.parent.mkdir(parents=True, exist_ok=True)

with open(artifact_path, 'wb') as f:
    pickle.dump(artifact, f)

print(f"Saved artifact: {artifact_path}")
print(f"Size: {artifact_path.stat().st_size / 1024:.1f} KB")
print(f"\nVerify by loading:")

with open(artifact_path, 'rb') as f:
    loaded = pickle.load(f)
print(f"  Keys: {list(loaded.keys())}")
print(f"  Model type: {type(loaded['model']).__name__}")
print(f"  Calibrator type: {type(loaded['calibrator']).__name__}")
print(f"  Feature count: {len(loaded['feature_cols'])}")
print(f"  Categorical cols: {loaded['categorical_cols']}")
print(f"  Model version: {loaded['model_version']}")

/var/folders/h2/8mlz42jd5v71w1ktjr4lp82r0000gn/T/ipykernel_71581/345415922.py:28: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  'training_date': pd.Timestamp.utcnow().isoformat(),


Saved artifact: ../models/model_artifact_v1.pkl
Size: 15.3 KB

Verify by loading:
  Keys: ['model', 'calibrator', 'feature_cols', 'categorical_cols', 'known_categories', 'model_version', 'training_metrics', 'best_params', 'training_date', 'n_train', 'n_val', 'n_test', 'best_iteration']
  Model type: Booster
  Calibrator type: IsotonicRegression
  Feature count: 42
  Categorical cols: ['archetype', 'market_location', 'gender', 'age_bracket']
  Model version: v1.0


### Save trained model artifact for inference

Pickle the calibrated final_model, isotonic calibrator, feature column names, categorical feature mappings, and evaluation metrics into a single artifact file. This artifact becomes the deployment package: downstream inference code loads this artifact to make consistent predictions on new data without retraining.

###  Score scaler sanity check

In [15]:
from inference.score_scaler import pd_to_score_batch
test_scores = pd_to_score_batch(test_pred_cal)

print(f"Score distribution on test set (n={len(test_scores)}):")
print(f"  Min:    {test_scores.min()}")
print(f"  P10:    {np.percentile(test_scores, 10):.0f}")
print(f"  P25:    {np.percentile(test_scores, 25):.0f}")
print(f"  Median: {np.median(test_scores):.0f}")
print(f"  P75:    {np.percentile(test_scores, 75):.0f}")
print(f"  P90:    {np.percentile(test_scores, 90):.0f}")
print(f"  Max:    {test_scores.max()}")

print(f"\n  FICO-band distribution:")
bands = [(800, 850, 'Exceptional'), (740, 799, 'Very Good'),
         (670, 739, 'Good'), (580, 669, 'Fair'), (300, 579, 'Poor')]
for low, high, label in bands:
    pct = ((test_scores >= low) & (test_scores <= high)).mean()
    count = ((test_scores >= low) & (test_scores <= high)).sum()
    print(f"    {label:>12} ({low}-{high}): {count:>4} ({pct:.1%})")

# Sanity: defaulters should skew low, non-defaulters high
print(f"\n  Score by actual outcome:")
print(f"    Non-defaulters median: {np.median(test_scores[y_test == 0]):.0f}")
print(f"    Defaulters median:     {np.median(test_scores[y_test == 1]):.0f}")
print(f"    Spread: {np.median(test_scores[y_test == 0]) - np.median(test_scores[y_test == 1]):.0f} points")

Score distribution on test set (n=1500):
  Min:    600
  P10:    652
  P25:    684
  Median: 740
  P75:    800
  P90:    800
  Max:    800

  FICO-band distribution:
     Exceptional (800-850):  687 (45.8%)
       Very Good (740-799):  136 (9.1%)
            Good (670-739):  405 (27.0%)
            Fair (580-669):  272 (18.1%)
            Poor (300-579):    0 (0.0%)

  Score by actual outcome:
    Non-defaulters median: 740
    Defaulters median:     673
    Spread: 67 points


In [17]:
print("Calibrated PD distribution on test set:")
print(f"  Min:    {test_pred_cal.min():.6f}")
print(f"  P5:     {np.percentile(test_pred_cal, 5):.6f}")
print(f"  P10:    {np.percentile(test_pred_cal, 10):.6f}")
print(f"  P25:    {np.percentile(test_pred_cal, 25):.6f}")
print(f"  Median: {np.median(test_pred_cal):.6f}")
print(f"  P75:    {np.percentile(test_pred_cal, 75):.6f}")
print(f"  P90:    {np.percentile(test_pred_cal, 90):.6f}")
print(f"  Max:    {test_pred_cal.max():.6f}")

# How many users sit at exactly the minimum calibrated PD?
min_pd = test_pred_cal.min()
at_floor = (test_pred_cal == min_pd).sum()
near_floor = (test_pred_cal <= min_pd * 1.01).sum()
print(f"\n  Users at exact min PD ({min_pd:.6f}): {at_floor} ({at_floor/len(test_pred_cal):.1%})")
print(f"  Users within 1% of min: {near_floor} ({near_floor/len(test_pred_cal):.1%})")

# Check raw (uncalibrated) predictions for comparison
raw_pred = final_model.predict(X_test)
print(f"\nRaw (uncalibrated) PD distribution:")
print(f"  Min:    {raw_pred.min():.6f}")
print(f"  P5:     {np.percentile(raw_pred, 5):.6f}")
print(f"  Median: {np.median(raw_pred):.6f}")
print(f"  Max:    {raw_pred.max():.6f}")

# Where does the calibrator clip kick in?
print(f"\nIsotonic calibrator X range during fit: [{calibrator.X_min_:.6f}, {calibrator.X_max_:.6f}]")
# print(f"Isotonic calibrator y range:            [{calibrator.y_min_:.6f}, {calibrator.y_max_:.6f}]")

Calibrated PD distribution on test set:
  Min:    0.017906
  P5:     0.017906
  P10:    0.017906
  P25:    0.017906
  Median: 0.057143
  P75:    0.158192
  P90:    0.260000
  Max:    0.500000

  Users at exact min PD (0.017906): 687 (45.8%)
  Users within 1% of min: 687 (45.8%)

Raw (uncalibrated) PD distribution:
  Min:    0.081631
  P5:     0.081631
  Median: 0.083452
  Max:    0.128437

Isotonic calibrator X range during fit: [0.081631, 0.128437]


### Diagnose probability calibration and floor clipping

The isotonic calibrator uses `out_of_bounds='clip'`, which maps out-of-sample PDs to the min/max seen during fitting. We check how many test samples hit this floor and inspect the raw vs calibrated PD distributions, percentiles, and distinct values. This reveals whether the calibrator has enough resolution and whether clipping is distorting the distribution. The goal is to ensure no excessive mass at the boundaries.

In [28]:
deeper_params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbose': -1,
    'random_state': 42,
    'class_weight': 'balanced',
    'bagging_freq': 5,
    # Trial 14 params but forced deeper
    'num_leaves': 13,
    'max_depth': 7,
    'learning_rate': 0.020,
    'min_child_samples': 47,
    'reg_alpha': 0.39,
    'reg_lambda': 0.10,
    'feature_fraction': 0.90,
    'bagging_fraction': 0.82,
}

deeper_model = lgb.train(
    deeper_params,
    lgb_train,
    num_boost_round=200,            # exactly 200, no early stopping
    valid_sets=[lgb_val],
    valid_names=['val'],
    callbacks=[lgb.log_evaluation(50)],
)

# Check resolution
raw = deeper_model.predict(X_test)
print(f"Deeper model raw PD range: [{raw.min():.4f}, {raw.max():.4f}]")
print(f"Distinct raw values: {len(np.unique(np.round(raw, 4)))}")
print(f"Val AUC: {roc_auc_score(y_val, deeper_model.predict(X_val)):.4f}")
print(f"Test AUC: {roc_auc_score(y_test, raw):.4f}")
print(f"\nPercentiles of raw PD:")
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"  P{p:>2}: {np.percentile(raw, p):.4f}")

# 2. Fit isotonic calibrator on val predictions
val_pred_raw = deeper_model.predict(X_val)
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(val_pred_raw, y_val)

# 3. Evaluate calibrated predictions on test
test_pred_raw = deeper_model.predict(X_test)
test_pred_cal = calibrator.predict(test_pred_raw)

# 4. KS statistic
def ks_statistic(y_true, y_pred):
    pos = np.sort(y_pred[y_true == 1])
    neg = np.sort(y_pred[y_true == 0])
    all_vals = np.sort(np.concatenate([pos, neg]))
    cdf_pos = np.searchsorted(pos, all_vals, side='right') / len(pos)
    cdf_neg = np.searchsorted(neg, all_vals, side='right') / len(neg)
    return float(np.max(np.abs(cdf_pos - cdf_neg)))

# 5. Approval rate at PD ≤ 5%
def approval_rate_at_threshold(y_true, y_pred, pd_threshold=0.05):
    approved = y_pred <= pd_threshold
    if approved.sum() == 0:
        return 0.0, 0.0
    approval_rate = approved.mean()
    default_rate_among_approved = y_true[approved].mean()
    return float(approval_rate), float(default_rate_among_approved)

approval_rate_5pct, default_among_approved = approval_rate_at_threshold(y_test, test_pred_cal, 0.05)

print(f"\n{'='*60}")
print(f"FINAL METRICS (CALIBRATED, ON TEST SET)")
print(f"{'='*60}")
print(f"  ROC-AUC:    {roc_auc_score(y_test, test_pred_cal):.4f}")
print(f"  PR-AUC:     {average_precision_score(y_test, test_pred_cal):.4f}")
print(f"  Brier:      {brier_score_loss(y_test, test_pred_cal):.4f}")
print(f"  LogLoss:    {log_loss(y_test, test_pred_cal):.4f}")
print(f"  KS:         {ks_statistic(y_test.values, test_pred_cal):.4f}")
print(f"  Approval @ PD≤5%: {approval_rate_5pct:.1%}")
print(f"  Default rate among approved: {default_among_approved:.1%}")

# 6. Calibration diagnostic: bin predictions, compare predicted vs actual
n_bins = 10
bin_edges = np.linspace(0, test_pred_cal.max(), n_bins + 1)
print(f"\n  Calibration check (predicted vs actual default rate):")
print(f"  {'Bin':>15}  {'Count':>6}  {'Pred':>8}  {'Actual':>8}")
for i in range(n_bins):
    mask = (test_pred_cal >= bin_edges[i]) & (test_pred_cal < bin_edges[i+1])
    if mask.sum() < 5:
        continue
    pred_mean = test_pred_cal[mask].mean()
    actual_mean = y_test[mask].mean()
    print(f"  [{bin_edges[i]:.3f},{bin_edges[i+1]:.3f}]  {mask.sum():>6}  {pred_mean:.4f}  {actual_mean:.4f}")

[50]	val's auc: 0.770031
[100]	val's auc: 0.765136
[150]	val's auc: 0.757121
[200]	val's auc: 0.754201
Deeper model raw PD range: [0.0136, 0.4644]
Distinct raw values: 930
Val AUC: 0.7542
Test AUC: 0.7621

Percentiles of raw PD:
  P 1: 0.0150
  P 5: 0.0177
  P10: 0.0194
  P25: 0.0252
  P50: 0.0552
  P75: 0.1387
  P90: 0.1913
  P95: 0.2329
  P99: 0.3229

FINAL METRICS (CALIBRATED, ON TEST SET)
  ROC-AUC:    0.7551
  PR-AUC:     0.1866
  Brier:      0.0763
  LogLoss:    0.2662
  KS:         0.4415
  Approval @ PD≤5%: 53.3%
  Default rate among approved: 2.2%

  Calibration check (predicted vs actual default rate):
              Bin   Count      Pred    Actual
  [0.000,0.044]     800  0.0217  0.0225
  [0.044,0.087]      42  0.0741  0.0952
  [0.131,0.175]     357  0.1479  0.1373
  [0.175,0.219]     213  0.1941  0.2066
  [0.262,0.306]      72  0.2779  0.2222


### Deeper tree exploration: attempt to improve score resolution

The calibrated PD distribution showed some bunching near the minimum, suggesting insufficient tree depth limited the model's ability to differentiate risk. We manually construct a deeper model (max_depth=7 instead of 5) with hyperparameters tuned from Optuna's Trial 14, train exactly 200 rounds without early stopping to maintain control, and re-calibrate on the validation set. This experiment tests whether deeper trees provide better score separation and resolution, enabling more granular risk binning downstream.

In [22]:
import pickle
from pathlib import Path
import pandas as pd

features_df = pd.read_parquet('../data/features_v1.parquet')

artifact = {
    'model': deeper_model,
    'calibrator': calibrator,
    'feature_cols': FEATURE_COLS,
    'categorical_cols': CATEGORICAL_FEATURES,
    'known_categories': {
        c: sorted(features_df[c].dropna().unique().tolist())
        for c in CATEGORICAL_FEATURES
    },
    'model_version': 'v1.0',
    'training_metrics': {
        'roc_auc_uncalibrated_test': 0.7680,
        'roc_auc_calibrated_test': float(roc_auc_score(y_test, test_pred_cal)),
        'pr_auc_test': float(average_precision_score(y_test, test_pred_cal)),
        'brier_test': float(brier_score_loss(y_test, test_pred_cal)),
        'logloss_test': float(log_loss(y_test, test_pred_cal)),
        'ks_test': ks_statistic(y_test.values, test_pred_cal),
        'approval_rate_at_5pct_pd': approval_rate_5pct,
        'default_rate_among_approved': default_among_approved,
    },
    'best_params': study.best_params,
    'training_date': pd.Timestamp.utcnow().isoformat(),
    'n_train': len(X_train),
    'n_val': len(X_val),
    'n_test': len(X_test),
    'best_iteration': deeper_model.best_iteration,
}

artifact_path = Path('../models/deeper_model_artifact_v1.pkl')
artifact_path.parent.mkdir(parents=True, exist_ok=True)

with open(artifact_path, 'wb') as f:
    pickle.dump(artifact, f)

print(f"Saved artifact: {artifact_path}")
print(f"Size: {artifact_path.stat().st_size / 1024:.1f} KB")
print(f"\nVerify by loading:")

with open(artifact_path, 'rb') as f:
    loaded = pickle.load(f)
print(f"  Keys: {list(loaded.keys())}")
print(f"  Model type: {type(loaded['model']).__name__}")
print(f"  Calibrator type: {type(loaded['calibrator']).__name__}")
print(f"  Feature count: {len(loaded['feature_cols'])}")
print(f"  Categorical cols: {loaded['categorical_cols']}")
print(f"  Model version: {loaded['model_version']}")

Saved artifact: ../models/deeper_model_artifact_v1.pkl
Size: 308.4 KB

Verify by loading:
  Keys: ['model', 'calibrator', 'feature_cols', 'categorical_cols', 'known_categories', 'model_version', 'training_metrics', 'best_params', 'training_date', 'n_train', 'n_val', 'n_test', 'best_iteration']
  Model type: Booster
  Calibrator type: IsotonicRegression
  Feature count: 42
  Categorical cols: ['archetype', 'market_location', 'gender', 'age_bracket']
  Model version: v1.0


/var/folders/h2/8mlz42jd5v71w1ktjr4lp82r0000gn/T/ipykernel_71581/331919152.py:28: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  'training_date': pd.Timestamp.utcnow().isoformat(),


### Save deeper model artifact for comparison

Serialize the deeper_model and its recalibrated isotonic calibrator to a separate artifact file. This allows A/B comparison in production: the original shallow model (better regularization) vs the deeper model (potentially better discrimination). Metrics are stored identically so deployment code can switch between versions without change.

In [30]:
import sys
sys.path.append('..') 
from inference.score_scaler import pd_to_score_batch
test_scores = pd_to_score_batch(test_pred_cal)

print(f"Score distribution on test set (n={len(test_scores)}):")
print(f"  Min:    {test_scores.min()}")
print(f"  P10:    {np.percentile(test_scores, 10):.0f}")
print(f"  P25:    {np.percentile(test_scores, 25):.0f}")
print(f"  Median: {np.median(test_scores):.0f}")
print(f"  P75:    {np.percentile(test_scores, 75):.0f}")
print(f"  P90:    {np.percentile(test_scores, 90):.0f}")
print(f"  Max:    {test_scores.max()}")

print(f"\n  FICO-band distribution:")
bands = [(800, 850, 'Exceptional'), (740, 799, 'Very Good'),
         (670, 739, 'Good'), (580, 669, 'Fair'), (300, 579, 'Poor')]
for low, high, label in bands:
    pct = ((test_scores >= low) & (test_scores <= high)).mean()
    count = ((test_scores >= low) & (test_scores <= high)).sum()
    print(f"    {label:>12} ({low}-{high}): {count:>4} ({pct:.1%})")

# Sanity: defaulters should skew low, non-defaulters high
print(f"\n  Score by actual outcome:")
print(f"    Non-defaulters median: {np.median(test_scores[y_test == 0]):.0f}")
print(f"    Defaulters median:     {np.median(test_scores[y_test == 1]):.0f}")
print(f"    Spread: {np.median(test_scores[y_test == 0]) - np.median(test_scores[y_test == 1]):.0f} points")

Score distribution on test set (n=1500):
  Min:    593
  P10:    673
  P25:    695
  Median: 824
  P75:    835
  P90:    835
  Max:    850

  FICO-band distribution:
     Exceptional (800-850):  800 (53.3%)
       Very Good (740-799):   42 (2.8%)
            Good (670-739):  570 (38.0%)
            Fair (580-669):   88 (5.9%)
            Poor (300-579):    0 (0.0%)

  Score by actual outcome:
    Non-defaulters median: 824
    Defaulters median:     694
    Spread: 130 points


### Validate deeper model score distribution and FICO-band alignment

Convert the deeper model's calibrated probabilities to FICO-style scores (300–850 scale) and inspect the distribution. Score separation between defaulters and non-defaulters should be clear, with defaulters clustering at lower bands (Poor/Fair) and non-defaulters at higher bands (Good/Very Good/Exceptional). This sanity check confirms that deeper tree structure improved score resolution and business interpretability without catastrophic overfitting.